In [ ]:
# these are the libraries that we need for our project
import pandas as pd

In [3]:
data_path = "data/Pan-India_Bus_Routes.csv"

In [ ]:
# ensure that all rows of a dataframe are displayed in the output. This will be useful for the value_counts() method
pd.set_option('display.max_rows', None)

bus_df = pd.read_csv(data_path, sep=",")

In [5]:
bus_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 35667 entries, 0 to 35666
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   From       35667 non-null  object
 1   To         35667 non-null  object
 2   Operator   35667 non-null  object
 3   Distance   35667 non-null  int64 
 4   Duration   35667 non-null  object
 5   Bus Type   35667 non-null  object
 6   Departure  35667 non-null  object
 7   Arrival    35667 non-null  object
dtypes: int64(1), object(7)
memory usage: 2.2+ MB


In [ ]:
# no Null values thus far. But there may be other ways, in which the data is incomplete
bus_df.isnull().sum()

From         0
To           0
Operator     0
Distance     0
Duration     0
Bus Type     0
Departure    0
Arrival      0
dtype: int64

In [13]:

with open("data/bus_starting_output.txt", "w") as f:
  print(bus_df["From"].value_counts(dropna=False), file=f)

In [14]:

with open("data/bus_destination_output.txt", "w") as f:
  print(bus_df["To"].value_counts(dropna=False), file=f)

Findings:
+ All origins and destinations are properly capitalized, and most of them have correct formatting. 
+ No rows have missing city names for origin or destination
+ Some places have "bypass" in their names. The number of routes with those places may be statistically insignificant
    + But it is still worthwhile to find out how to treat them properly


In [17]:
with open("data/bus_operator_output.txt", "w") as f:
  print(bus_df["Operator"].value_counts(dropna=False), file=f)

For operators, it's more complicated:
+ Different operators have very similar names
    + i.e. sharma travels (nanded)-(chintamani) and sharma travels (nanded)-devgiri express-VT
    + Are they the parts of the same company? Or should they be treated as separate entities?
    + there are many such cases, so this is a more important problem than the "bypass" problem mentioned above.

In [19]:
with open("data/bus_distance_output.txt", "w") as f:
  print(bus_df["Distance"].value_counts(dropna=False, normalize=True), file=f)

Distance between cities measured in kilometers - pretty straightforward. Distances of hundreds of kilometers are common, but that's not surprising.

In [23]:
with open("data/travel_duration_output.txt", "w") as f:
  print(bus_df["Duration"].value_counts(dropna=False), file=f)

The original dataset claims that duration is in the format "hh:mm:ss", but that is wrong. The second number in the timestamp is never greater than 23, implying that it is the hour, and not the minute. Moreover, distances between cities are so long, that it is impossible to clear them in mere hours. Therefore, the more appropriate timestamp format would be "d:hh:mm".

Next step, convert the "Duration" column into a proper timestamp recognized by pandas

In [24]:
with open("data/bus_type_output.txt", "w") as f:
  print(bus_df["Bus Type"].value_counts(dropna=False), file=f)

This is the most difficult column to clean. We must split it into several different columns, each containing the following information:
+ sleeper vs non-sleeper - represents the extent, to which this bus is designed for sleeping
+ brand of bus
+ seat configuration(2+2, 1+2, etc.) - represents # of seats per row
+ AC vs non-AC - presence of air conditioning
+ miscellaneous - descriptions that do not fit into any of the above categories